# 11 - 训练循环 (AI Infra 视角)

本节从 **工程实现** 角度理解 LLM 训练循环：
- 基础训练循环结构
- 交叉熵损失
- 梯度累积
- 混合精度训练
- 梯度裁剪
- 检查点管理
- 训练监控指标

> 参考 nanochat/scripts/base_train.py

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

## 1. 训练循环核心 (30秒版)

```
for step in range(num_iterations):
    # 1. 前向传播
    loss = model(x, y)
    
    # 2. 反向传播
    loss.backward()
    
    # 3. 梯度裁剪
    clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    # 4. 更新参数
    optimizer.step()
    
    # 5. 清零梯度
    optimizer.zero_grad()
```

实际训练还需要：梯度累积、混合精度、学习率调度、检查点、日志...

## 2. 交叉熵损失

In [2]:
# LLM 的损失函数：预测下一个 token

vocab_size = 50304
B, T = 4, 1024

# 模型输出 logits
logits = torch.randn(B, T, vocab_size)  # 模型预测
targets = torch.randint(0, vocab_size, (B, T))  # 正确答案

# 方法 1: 直接用 F.cross_entropy
loss = F.cross_entropy(logits.view(B * T, vocab_size), targets.view(B * T))
print(f"Loss: {loss.item():.4f}")

# 随机初始化时的期望 loss
expected = math.log(vocab_size)
print(f"随机初始化期望 loss: {expected:.4f}")
print(f"\n如果训练初始 loss 远大于 {expected:.1f}，说明初始化有问题")

Loss: 11.3195
随机初始化期望 loss: 10.8258

如果训练初始 loss 远大于 10.8，说明初始化有问题


### Loss 和 BPB (Bits Per Byte) 的关系

```
Loss = cross_entropy (nats per token)
BPB  = loss / ln(2) / avg_bytes_per_token

BPB 是更公平的比较指标，因为不同分词器的 token 粒度不同
```

In [3]:
# nanochat 用 BPB (Bits Per Byte) 评估
def loss_to_bpb(loss, avg_bytes_per_token=3.5):
    """将 cross entropy loss 转换为 BPB"""
    return loss / math.log(2) / avg_bytes_per_token

# 示例
for loss_val in [10.8, 5.0, 3.0, 2.5]:
    bpb = loss_to_bpb(loss_val)
    ppl = math.exp(loss_val)
    print(f"Loss: {loss_val:.1f} | BPB: {bpb:.2f} | Perplexity: {ppl:.1f}")

Loss: 10.8 | BPB: 4.45 | Perplexity: 49020.8
Loss: 5.0 | BPB: 2.06 | Perplexity: 148.4
Loss: 3.0 | BPB: 1.24 | Perplexity: 20.1
Loss: 2.5 | BPB: 1.03 | Perplexity: 12.2


## 3. 梯度累积 (重要!)

**问题**: 想用大 batch (512K tokens) 但显存装不下

**解决**: 多次小 batch forward/backward，累积梯度后再更新

```
期望 total_batch_size = 524288 tokens
单卡 device_batch_size = 32
序列长度 seq_len = 2048
8 张 GPU

每次 forward 处理: 32 × 2048 = 65536 tokens
所有 GPU 一次: 65536 × 8 = 524288 tokens
梯度累积步数: 524288 / 524288 = 1

如果只有 1 张 GPU:
梯度累积步数: 524288 / 65536 = 8
```

In [4]:
# 梯度累积的计算
def compute_grad_accum(total_batch_size, device_batch_size, seq_len, world_size):
    tokens_per_fwdbwd = device_batch_size * seq_len
    world_tokens_per_fwdbwd = tokens_per_fwdbwd * world_size
    grad_accum_steps = total_batch_size // world_tokens_per_fwdbwd
    
    print(f"期望 batch size: {total_batch_size:,} tokens")
    print(f"每次 forward:    {tokens_per_fwdbwd:,} tokens/GPU")
    print(f"所有 GPU 一次:   {world_tokens_per_fwdbwd:,} tokens")
    print(f"梯度累积步数:    {grad_accum_steps}")
    return grad_accum_steps

# nanochat 默认配置
print("=== 8×H100 ===")
compute_grad_accum(524288, device_batch_size=32, seq_len=2048, world_size=8)

print("\n=== 单卡 ===")
compute_grad_accum(524288, device_batch_size=32, seq_len=2048, world_size=1)

=== 8×H100 ===
期望 batch size: 524,288 tokens
每次 forward:    65,536 tokens/GPU
所有 GPU 一次:   524,288 tokens
梯度累积步数:    1

=== 单卡 ===
期望 batch size: 524,288 tokens
每次 forward:    65,536 tokens/GPU
所有 GPU 一次:   65,536 tokens
梯度累积步数:    8


8

In [5]:
# 梯度累积实现

model = nn.Linear(64, 64)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
grad_accum_steps = 4

# ❌ 错误写法: loss 没有缩放
bad_code = '''
for micro_step in range(grad_accum_steps):
    loss = model(x).sum()
    loss.backward()  # 梯度会累加，但 loss 没除以 accum_steps
optimizer.step()
'''

# ✅ 正确写法: loss 除以 accum_steps
print("正确的梯度累积:")
print()
for micro_step in range(grad_accum_steps):
    x = torch.randn(8, 64)
    loss = model(x).sum()
    loss = loss / grad_accum_steps  # 关键: 除以累积步数!
    loss.backward()  # .backward() 默认累加梯度
    print(f"  micro_step {micro_step}: loss={loss.item():.4f}")

# 累积完了再更新
optimizer.step()
optimizer.zero_grad()
print("\noptimizer.step() 使用累积的梯度更新参数")

正确的梯度累积:

  micro_step 0: loss=0.8654
  micro_step 1: loss=-4.8721
  micro_step 2: loss=-0.6647
  micro_step 3: loss=2.1486

optimizer.step() 使用累积的梯度更新参数


### 为什么要除以 grad_accum_steps?

```
backward() 默认是梯度累加 (不是覆盖)

不除:  grad = g0 + g1 + g2 + g3           → 4 倍大的梯度
除了:  grad = g0/4 + g1/4 + g2/4 + g3/4   → 等效于 4 倍 batch 的平均梯度

数学上等价于:
  grad_accum_steps=4, batch=8  ≡  一次 batch=32
```

In [6]:
# 验证: 梯度累积 ≡ 大 batch

torch.manual_seed(42)
model_a = nn.Linear(16, 16, bias=False)
model_b = nn.Linear(16, 16, bias=False)
model_b.load_state_dict(model_a.state_dict())  # 相同初始权重

# 准备 4 个小 batch
torch.manual_seed(0)
batches = [torch.randn(8, 16) for _ in range(4)]
big_batch = torch.cat(batches, dim=0)  # 合并成大 batch

# 方法 A: 大 batch 一次算
loss_a = F.mse_loss(model_a(big_batch), torch.zeros(32, 16))
loss_a.backward()

# 方法 B: 梯度累积
for batch in batches:
    loss_b = F.mse_loss(model_b(batch), torch.zeros(8, 16))
    loss_b = loss_b / 4  # 除以累积步数
    loss_b.backward()

# 对比梯度
diff = (model_a.weight.grad - model_b.weight.grad).abs().max().item()
print(f"梯度最大差异: {diff:.10f}")
print(f"梯度完全一致!" if diff < 1e-6 else "梯度有差异!")

梯度最大差异: 0.0000000149
梯度完全一致!


## 4. 混合精度训练

```
前向/反向: BF16 (快, 省显存)
参数更新: FP32 (精度高)

PyTorch 通过 autocast 自动管理:
  矩阵乘法 → BF16 (计算密集型)
  LayerNorm, Softmax → FP32 (数值敏感型)
```

In [7]:
# 混合精度训练
from contextlib import nullcontext

device_type = "cuda" if torch.cuda.is_available() else "cpu"

# 自动选择 autocast context
if device_type == "cuda":
    autocast_ctx = torch.amp.autocast(device_type=device_type, dtype=torch.bfloat16)
else:
    autocast_ctx = nullcontext()  # CPU 不用 autocast

print(f"Device: {device_type}")
print(f"Autocast: {'BF16' if device_type == 'cuda' else '禁用 (CPU)'}")
print()
print("autocast 自动处理的精度:")
print("  nn.Linear (matmul)  → BF16  (快)")
print("  F.softmax           → FP32  (精确)")
print("  F.layer_norm        → FP32  (精确)")
print("  F.cross_entropy     → FP32  (精确)")

Device: cuda
Autocast: BF16

autocast 自动处理的精度:
  nn.Linear (matmul)  → BF16  (快)
  F.softmax           → FP32  (精确)
  F.layer_norm        → FP32  (精确)
  F.cross_entropy     → FP32  (精确)


### BF16 vs FP16

```
nanochat 用 BF16 (不是 FP16)

BF16 优势:
  - 范围和 FP32 一样大 (8 位指数)
  - 不需要 GradScaler (不容易溢出)
  - 代码更简单

FP16 需要 GradScaler:
  scaler = GradScaler()
  scaler.scale(loss).backward()
  scaler.step(optimizer)
  scaler.update()
  
BF16 直接用:
  with autocast(dtype=torch.bfloat16):
      loss = model(x, y)
  loss.backward()       # 不需要 scaler!
  optimizer.step()
```

## 5. 梯度裁剪

**问题**: 梯度偶尔会非常大 (gradient spike)，导致训练不稳定

**解决**: 限制梯度的范数不超过阈值

```
grad_norm = ||all_grads||₂        # 计算所有参数梯度的 L2 范数

if grad_norm > max_norm:
    scale = max_norm / grad_norm
    all_grads *= scale             # 等比例缩小
```

In [8]:
# 梯度裁剪演示
model = nn.Linear(64, 64)
x = torch.randn(8, 64)
loss = model(x).sum()
loss.backward()

# 裁剪前
grad_norm_before = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
print(f"裁剪前 grad_norm: {grad_norm_before:.4f}")

# 重新计算梯度
model.zero_grad()
loss = model(x).sum()
loss.backward()

# 裁剪
max_norm = 1.0
grad_norm_after = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

# 验证裁剪后的范数
actual_norm = torch.cat([p.grad.flatten() for p in model.parameters()]).norm()
print(f"裁剪后 grad_norm: {actual_norm:.4f}")
print(f"max_norm: {max_norm}")
print(f"\nclip_grad_norm_ 返回的是裁剪前的 grad_norm: {grad_norm_after:.4f}")

裁剪前 grad_norm: 194.7285
裁剪后 grad_norm: 1.0000
max_norm: 1.0

clip_grad_norm_ 返回的是裁剪前的 grad_norm: 194.7285


### 梯度裁剪的注意事项

```
1. clip_grad_norm_ 返回的是裁剪前的 grad_norm (用于监控)
2. 在 optimizer.step() 之前调用
3. 在梯度累积完成之后调用 (不是每个 micro step)
4. nanochat 默认 grad_clip = 1.0

监控 grad_norm:
  正常训练: grad_norm 在 0.1-10 之间波动
  出现 spike: grad_norm 突然跳到 100+ → 被裁剪
  持续很大: 可能学习率太大或数据有问题
```

## 6. 完整训练循环 (nanochat 风格)

In [9]:
# 完整训练循环 (简化版，体现核心逻辑)

def train(
    model,
    train_loader,
    optimizer,
    num_iterations,
    grad_accum_steps=1,
    grad_clip=1.0,
    warmup_steps=100,
    device_type="cpu",
    max_lr=1e-3,
):
    autocast_ctx = (
        torch.amp.autocast(device_type=device_type, dtype=torch.bfloat16)
        if device_type == "cuda" else nullcontext()
    )
    
    model.train()
    
    for step in range(num_iterations):
        t0 = time.time()
        
        # --- 学习率调度 ---
        lr = get_lr(step, warmup_steps, num_iterations, max_lr, min_lr=max_lr * 0.1)
        for group in optimizer.param_groups:
            group['lr'] = lr
        
        # --- 梯度累积 ---
        for micro_step in range(grad_accum_steps):
            x, y = next(train_loader)
            
            with autocast_ctx:
                loss = model(x, y)  # 前向
            
            loss = loss / grad_accum_steps  # 缩放 loss
            loss.backward()  # 反向 (梯度累加)
        
        # --- 梯度裁剪 (累积完成后) ---
        if grad_clip > 0.0:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        # --- 更新参数 ---
        optimizer.step()
        model.zero_grad(set_to_none=True)  # 比 zero_grad() 更快
        
        # --- 日志 ---
        dt = time.time() - t0
        if step % 10 == 0:
            print(f"step {step:05d} | loss: {loss.item() * grad_accum_steps:.4f} | "
                  f"lr: {lr:.2e} | dt: {dt*1000:.0f}ms")

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr=0):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

print("完整训练循环的执行顺序:")
print("  1. 调整学习率")
print("  2. 梯度累积 (多次 forward + backward)")
print("  3. 梯度裁剪")
print("  4. optimizer.step()")
print("  5. zero_grad()")

完整训练循环的执行顺序:
  1. 调整学习率
  2. 梯度累积 (多次 forward + backward)
  3. 梯度裁剪
  4. optimizer.step()
  5. zero_grad()


## 7. zero_grad 的细节

In [ ]:
model = nn.Linear(8, 8)
x = torch.randn(2, 8)

# 第一次 backward
model(x).sum().backward()
print(f"第一次 backward 后 grad: {model.weight.grad[0,:4]}")

# 第二次 backward (不清零，梯度会累加!)
model(x).sum().backward()
print(f"第二次 backward 后 grad: {model.weight.grad[0,:4]}  ← 翻倍了!")

# 清零方式 1: zero_grad(set_to_none=False) — grad 变成全 0 tensor
model.zero_grad(set_to_none=False)
print(f"zero_grad(set_to_none=False) 后: {model.weight.grad[0,:4]}")

# 清零方式 2: zero_grad() — 新版 PyTorch 默认 set_to_none=True
model(x).sum().backward()
model.zero_grad()  # 等价于 model.zero_grad(set_to_none=True)
print(f"zero_grad() 后: {model.weight.grad}")
print()
print("两种方式对比:")
print("  set_to_none=False: grad 变成全零 tensor (仍然占显存)")
print("  set_to_none=True (默认): grad 设为 None (释放显存，推荐)")
print()
print("注意: 新版 PyTorch 的 zero_grad() 默认就是 set_to_none=True")

## 8. Prefetch: 数据加载与计算重叠

nanochat 在 GPU 做 forward/backward 的同时，CPU 加载下一个 batch。

```python
# nanochat base_train.py 中的关键行:
for micro_step in range(grad_accum_steps):
    loss = model(x, y)            # GPU 在计算
    loss.backward()
    x, y = next(train_loader)     # CPU 加载下一个 batch
                                  # ↑ GPU 还在算，CPU 已经开始准备了
```

配合 `pin_memory + non_blocking` 实现异步传输:

```
GPU:  [forward+backward batch0][forward+backward batch1]...
CPU:   [加载 batch1] [传输→GPU]  [加载 batch2] [传输→GPU]
        ↑ 重叠执行
```

## 9. torch.compile

nanochat 使用 `torch.compile` 加速模型。

```python
# nanochat 的用法:
orig_model = model
model = torch.compile(model, dynamic=False)
#                               ↑ 输入形状固定，编译更激进

# 保存 orig_model (未编译版) 用于:
#   1. 保存 checkpoint (编译后的 state_dict 可能有问题)
#   2. 推理/评估 (输入形状可能变化)
```

In [13]:
# torch.compile 基本效果演示
import torch
import torch.nn as nn

# 检测 torch.compile 是否能在当前环境工作
def check_compile_available():
    """检测 torch.compile 是否可用，返回 (可用, 设备) 或 (不可用, 原因)"""
    for device in (["cuda"] if torch.cuda.is_available() else []) + ["cpu"]:
        try:
            @torch.compile
            def _test(x):
                return x + 1
            _test(torch.tensor([1.0], device=device))
            torch._dynamo.reset()
            return True, device
        except Exception as e:
            torch._dynamo.reset()
            last_error = str(e)
    return False, last_error

compile_ok, compile_info = check_compile_available()

# 定义一个简单模型
class SimpleModel(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.linear1 = nn.Linear(dim, dim * 4)
        self.linear2 = nn.Linear(dim * 4, dim)
    
    def forward(self, x):
        x = self.linear1(x)
        x = x * torch.sigmoid(x)  # SiLU/Swish，多个小 op 可以融合
        x = self.linear2(x)
        return x

if compile_ok:
    device = compile_info
    model_eager = SimpleModel().to(device)
    model_compiled = torch.compile(SimpleModel().to(device))
    x = torch.randn(32, 512, device=device)

    print(f"使用设备: {device}")
    print("第一次调用触发编译（会比较慢）...")
    t0 = time.time()
    _ = model_compiled(x)
    print(f"首次编译耗时: {time.time() - t0:.2f}s\n")

    if device == "cuda":
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(1000):
            _ = model_eager(x)
        torch.cuda.synchronize()
        eager_time = time.time() - t0

        t0 = time.time()
        for _ in range(1000):
            _ = model_compiled(x)
        torch.cuda.synchronize()
        compiled_time = time.time() - t0
        n_iter = 1000
    else:
        t0 = time.time()
        for _ in range(100):
            _ = model_eager(x)
        eager_time = time.time() - t0

        t0 = time.time()
        for _ in range(100):
            _ = model_compiled(x)
        compiled_time = time.time() - t0
        n_iter = 100

    print(f"未编译: {eager_time*1000:.1f}ms ({n_iter}次)")
    print(f"编译后: {compiled_time*1000:.1f}ms ({n_iter}次)")
    print(f"加速比: {eager_time/compiled_time:.2f}x")
else:
    print("⚠ torch.compile 在当前环境不可用:")
    print(f"  CUDA: {'可用' if torch.cuda.is_available() else '不可用'} (需要 Triton)")
    print(f"  CPU:  需要 C++ 编译器 (cl.exe / gcc)")
    print(f"\n  错误信息: {compile_info[:100]}...")
    print()
    print("这不影响理解 torch.compile 的原理。在 Linux + GPU 环境下效果:")
    print("  典型加速比: 1.5x - 2.5x")
    print("  主要来源: kernel 融合 (减少显存读写次数)")
    print("  例如 SiLU(x) = x * sigmoid(x) 的 3 次显存读写 → 1 次")

⚠ torch.compile 在当前环境不可用:
  CUDA: 可用 (需要 Triton)
  CPU:  需要 C++ 编译器 (cl.exe / gcc)

  错误信息: RuntimeError: Compiler: cl is not found.

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (pl...

这不影响理解 torch.compile 的原理。在 Linux + GPU 环境下效果:
  典型加速比: 1.5x - 2.5x
  主要来源: kernel 融合 (减少显存读写次数)
  例如 SiLU(x) = x * sigmoid(x) 的 3 次显存读写 → 1 次


### 方法 1: dynamo.explain — 查看计算图信息

In [14]:
# dynamo.explain: 查看 torch.compile 追踪到的计算图信息
import torch._dynamo as dynamo

# 定义一个包含多个可融合操作的函数
def swish_layernorm(x, weight, bias):
    x = x * torch.sigmoid(x)       # SiLU/Swish
    x = F.layer_norm(x, [x.shape[-1]])  # LayerNorm
    x = x * weight + bias          # scale + shift
    return x

x = torch.randn(8, 256)
weight = torch.randn(256)
bias = torch.randn(256)

# explain 会告诉你编译器能追踪多少操作、有没有 graph break
explanation = dynamo.explain(swish_layernorm)(x, weight, bias)
print(explanation)
print()
print("关键信息:")
print("  graph_count: 计算图被切成了几段 (1 = 没有 graph break，最好)")
print("  break_reasons: 导致计算图中断的原因 (越少越好)")

Graph Count: 1
Graph Break Count: 0
Op Count: 5
Break Reasons:
Ops per Graph:
  Ops 1:
    <built-in method sigmoid of type object at 0x00007FFC00AB6DC0>
    <built-in function mul>
    <function layer_norm at 0x000001AE38DAD1C0>
    <built-in function mul>
    <built-in function add>
Out Guards:
  Guard 1:
    Name: ''
    Source: shape_env
    Create Function: SHAPE_ENV
    Guard Types: None
    Code List: None
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 2:
    Name: ''
    Source: global
    Create Function: DETERMINISTIC_ALGORITHMS
    Guard Types: None
    Code List: None
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 3:
    Name: ''
    Source: global
    Create Function: GRAD_MODE
    Guard Types: None
    Code List: None
    Object Weakref: None
    Guarded Class Weakref: None
  Guard 4:
    Name: ''
    Source: global
    Create Function: DEFAULT_DEVICE
    Guard Types: ['DEFAULT_DEVICE']
    Code List: ['utils_device.CURRENT_DEVICE == No

### 方法 2: 查看生成的 Triton kernel 代码

设置 `output_code = True` 后，torch.compile 会打印编译产出的实际 GPU kernel 代码。
可以看到多个操作被融合成一个 kernel。

In [15]:
# 查看 torch.compile 生成的代码
import os

if compile_ok:
    torch._dynamo.reset()  # 清除之前的编译缓存
    
    # 设置日志，让编译产出的代码打印到 notebook
    os.environ["TORCH_LOGS"] = "output_code"
    torch._logging.set_logs(output_code=True)
    
    # 定义一个简单函数：3 个操作可以融合
    @torch.compile
    def fused_ops(x):
        x = x + 1          # op 1
        x = x * 2          # op 2
        x = torch.relu(x)  # op 3
        return x
    
    device = compile_info
    print("=== 触发编译，观察下方输出中生成的 kernel 代码 ===")
    if device == "cuda":
        print("(关注 @triton.jit 开头的函数，那就是融合后的 GPU kernel)")
    else:
        print("(CPU 模式下生成的是 C++ 代码而非 Triton kernel)")
    print()
    
    result = fused_ops(torch.randn(8, 256, device=device))
    print(f"\n输出 shape: {result.shape}, device: {result.device}")
    
    # 恢复日志设置
    torch._logging.set_logs(output_code=False)
    os.environ.pop("TORCH_LOGS", None)
else:
    print("⚠ torch.compile 不可用，跳过代码生成演示")
    print()
    print("在可用环境下，设置 output_code=True 后能看到:")
    print()
    print("  1. Triton kernel (GPU):")
    print("     @triton.jit")
    print("     def triton_poi_fused_add_mul_relu(in_ptr0, out_ptr0, xnumel, ...):")
    print("         # 3 个 op (add, mul, relu) 被融合成 1 个 kernel!")
    print("         tmp0 = tl.load(in_ptr0 + xoffset)")
    print("         tmp1 = tmp0 + 1        # add")
    print("         tmp2 = tmp1 * 2        # mul")
    print("         tmp3 = triton_helpers.maximum(0, tmp2)  # relu")
    print("         tl.store(out_ptr0 + xoffset, tmp3)")
    print()
    print("  2. C++ kernel (CPU):")
    print("     void kernel(const float* in, float* out, long n) {")
    print("         for (long i = 0; i < n; i++) {")
    print("             float tmp = (in[i] + 1.0f) * 2.0f;")
    print("             out[i] = tmp > 0 ? tmp : 0;  // fused!")
    print("         }")
    print("     }")
    print()
    print("关键: 3 次显存读写 → 1 次 (数据只需从显存读一次、写一次)")

⚠ torch.compile 不可用，跳过代码生成演示

在可用环境下，设置 output_code=True 后能看到:

  1. Triton kernel (GPU):
     @triton.jit
     def triton_poi_fused_add_mul_relu(in_ptr0, out_ptr0, xnumel, ...):
         # 3 个 op (add, mul, relu) 被融合成 1 个 kernel!
         tmp0 = tl.load(in_ptr0 + xoffset)
         tmp1 = tmp0 + 1        # add
         tmp2 = tmp1 * 2        # mul
         tmp3 = triton_helpers.maximum(0, tmp2)  # relu
         tl.store(out_ptr0 + xoffset, tmp3)

  2. C++ kernel (CPU):
     void kernel(const float* in, float* out, long n) {
         for (long i = 0; i < n; i++) {
             float tmp = (in[i] + 1.0f) * 2.0f;
             out[i] = tmp > 0 ? tmp : 0;  // fused!
         }
     }

关键: 3 次显存读写 → 1 次 (数据只需从显存读一次、写一次)


### 方法 3: 查看磁盘上的编译缓存文件

torch.compile 会把生成的 kernel 代码缓存到磁盘。你可以直接查看这些文件。

In [ ]:
# 查看编译缓存目录中的生成文件
import tempfile
import glob as glob_module
import getpass

if compile_ok:
    # torch.compile 的缓存位置
    username = getpass.getuser()
    cache_dirs = [
        os.path.join(tempfile.gettempdir(), f"torchinductor_{username}"),
        os.path.expanduser("~/.triton/cache"),
    ]

    for cache_dir in cache_dirs:
        if os.path.exists(cache_dir):
            py_files = glob_module.glob(os.path.join(cache_dir, "**/*.py"), recursive=True)
            print(f"缓存目录: {cache_dir}")
            print(f"  生成的 .py 文件数量: {len(py_files)}")
            if py_files:
                newest = max(py_files, key=os.path.getmtime)
                print(f"  最新文件: {newest}")
                print(f"  --- 前 30 行 ---")
                with open(newest, 'r', encoding='utf-8', errors='replace') as f:
                    lines = f.readlines()[:30]
                    for line in lines:
                        print(f"  {line}", end='')
                total = len(open(newest, encoding='utf-8', errors='replace').readlines())
                print(f"\n  --- (共 {total} 行) ---")
            print()
        else:
            print(f"缓存目录不存在: {cache_dir}")
            print(f"  (运行 torch.compile 后才会生成)")
            print()

    print("提示: 你可以直接打开这些 .py 文件查看完整的生成代码")
    print("  其中 @triton.jit 装饰的函数就是融合后的 GPU kernel")
else:
    print("⚠ torch.compile 未运行，没有缓存文件")
    print()
    print("缓存文件的典型位置:")
    print(f"  Windows: %TEMP%\\torchinductor_<username>\\")
    print(f"  Linux:   /tmp/torchinductor_<username>/")
    print(f"  Triton:  ~/.triton/cache/")
    print()
    print("每个缓存文件包含一个完整的 Python 文件，其中:")
    print("  - 导入 triton 库")
    print("  - @triton.jit 装饰的 kernel 函数 (实际 GPU 代码)")
    print("  - call() 函数 (启动 kernel 的入口)")

## 10. 检查点管理

In [ ]:
# 保存检查点: 完整恢复训练所需的所有状态

def save_checkpoint(checkpoint_dir, step, model, optimizers, dataloader_state, extra):
    """
    需要保存的内容:
      1. 模型参数
      2. 优化器状态 (m, v, step 计数)
      3. 数据加载器状态 (读到哪里了)
      4. 训练循环状态 (step, loss 等)
    """
    checkpoint = {
        'model': model.state_dict(),
        'optimizers': [opt.state_dict() for opt in optimizers],
        'dataloader': dataloader_state,
        'step': step,
        **extra,
    }
    path = f"{checkpoint_dir}/step_{step:06d}.pt"
    torch.save(checkpoint, path)
    return path

def load_checkpoint(path, model, optimizers):
    checkpoint = torch.load(path, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    for opt, state in zip(optimizers, checkpoint['optimizers']):
        opt.load_state_dict(state)
    return checkpoint

print("检查点必须保存的内容:")
print("  模型参数     → 恢复模型")
print("  优化器状态   → 恢复动量 (不保存则 warmup 要重来)")
print("  dataloader   → 恢复数据进度 (不保存则重复训练数据)")
print("  step         → 恢复学习率调度")

### nanochat 的检查点结构

```
base_checkpoints/d20/
├── step_000000/
│   ├── model.pt              # 模型参数
│   ├── optimizer_rank0.pt    # 优化器状态 (每个 rank 不同)
│   ├── optimizer_rank1.pt
│   └── meta.json             # 元数据 (step, loss, 配置等)
├── step_000250/
│   └── ...
└── step_final/
    └── ...

分布式训练时:
  - 模型参数: 所有 rank 相同，只存一份
  - 优化器状态: ZeRO-2 分片，每个 rank 存自己的那份
```

## 11. 训练监控指标

In [16]:
# nanochat 监控的指标

def compute_metrics(step, loss, dt, total_batch_size, num_flops_per_token,
                    num_iterations, grad_norm=None, ddp_world_size=1):
    """计算训练监控指标"""
    # 训练进度
    pct_done = 100 * step / num_iterations
    
    # 吞吐量
    tok_per_sec = total_batch_size / dt
    
    # MFU (Model FLOPs Utilization)
    flops_per_sec = num_flops_per_token * total_batch_size / dt
    # H100 BF16 理论峰值: 989 TFLOPS
    promised_flops = 989e12 * ddp_world_size
    mfu = 100 * flops_per_sec / promised_flops
    
    print(f"step {step:05d}/{num_iterations} ({pct_done:.1f}%)")
    print(f"  loss:      {loss:.4f}")
    print(f"  dt:        {dt*1000:.0f}ms")
    print(f"  tok/sec:   {tok_per_sec:,.0f}")
    print(f"  MFU:       {mfu:.1f}%")
    if grad_norm is not None:
        print(f"  grad_norm: {grad_norm:.4f}")

# 模拟
compute_metrics(
    step=500, loss=4.2, dt=0.5,
    total_batch_size=524288, num_flops_per_token=3.4e9,
    num_iterations=5000, grad_norm=0.87, ddp_world_size=8
)

step 00500/5000 (10.0%)
  loss:      4.2000
  dt:        500ms
  tok/sec:   1,048,576
  MFU:       45.1%
  grad_norm: 0.8700


### MFU (Model FLOPs Utilization)

```
MFU = 实际计算速度 / GPU 理论峰值

MFU 50-60%: 很好
MFU 30-40%: 一般 (可能有瓶颈)
MFU < 20%:  有问题 (数据加载慢? 通信瓶颈?)

H100 SXM BF16 理论峰值: 989 TFLOPS
A100 SXM BF16 理论峰值: 312 TFLOPS
```

### EMA 平滑 Loss

原始 loss 波动大，用 EMA (指数移动平均) 平滑后更好看趋势。

In [ ]:
import matplotlib.pyplot as plt

# 模拟 loss 曲线
torch.manual_seed(42)
raw_losses = [10.0 * math.exp(-0.005 * i) + 0.5 * torch.randn(1).item() for i in range(500)]

# EMA 平滑 (nanochat 用的方法)
ema_beta = 0.9
smooth_loss = 0
smooth_losses = []
for i, loss in enumerate(raw_losses):
    smooth_loss = ema_beta * smooth_loss + (1 - ema_beta) * loss
    debiased = smooth_loss / (1 - ema_beta ** (i + 1))  # bias correction
    smooth_losses.append(debiased)

plt.figure(figsize=(10, 4))
plt.plot(raw_losses, alpha=0.3, label='Raw loss')
plt.plot(smooth_losses, label='EMA smoothed (β=0.9)')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Loss Smoothing with EMA')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("EMA 公式:")
print("  smooth = β * smooth + (1-β) * loss")
print("  debiased = smooth / (1 - β^step)")
print(f"\n  β=0.9: 约看最近 10 步的平均")
print(f"  β=0.99: 约看最近 100 步的平均")

## 12. nanochat 训练循环全景

```python
# nanochat/scripts/base_train.py 核心逻辑

# --- 初始化 ---
model = GPT(config)                              # 创建模型
model.init_weights()                              # 初始化权重
model = torch.compile(model, dynamic=False)       # 编译加速
optimizers = model.setup_optimizers(...)           # AdamW + Muon
x, y = next(train_loader)                         # 预取第一个 batch

# --- 主循环 ---
while True:
    last_step = (step == num_iterations)
    
    # 定期评估
    if step % eval_every == 0:
        model.eval()
        val_bpb = evaluate_bpb(model, val_loader)
        model.train()
    
    # 保存检查点
    if last_step or step % save_every == 0:
        save_checkpoint(...)
    
    if last_step:
        break
    
    # === 单步训练 ===
    t0 = time.time()
    
    # 梯度累积
    for micro_step in range(grad_accum_steps):
        with autocast(dtype=torch.bfloat16):
            loss = model(x, y)
        (loss / grad_accum_steps).backward()
        x, y = next(train_loader)                 # prefetch!
    
    # 梯度裁剪
    grad_norm = clip_grad_norm_(model.parameters(), 1.0)
    
    # 更新学习率
    lrm = get_lr_multiplier(step)
    for opt in optimizers:
        for group in opt.param_groups:
            group['lr'] = group['initial_lr'] * lrm
    
    # 更新参数
    for opt in optimizers:
        opt.step()
    model.zero_grad(set_to_none=True)
    
    dt = time.time() - t0
    step += 1
```

### 执行顺序图

```
┌──────────────────────────────────────────────────────┐
│  评估 (eval_every 步)                                │
│  保存检查点 (save_every 步)                           │
├──────────────────────────────────────────────────────┤
│  梯度累积循环 (grad_accum_steps 次):                  │
│    autocast → forward → loss/accum → backward        │
│    prefetch next batch                               │
├──────────────────────────────────────────────────────┤
│  clip_grad_norm_                                     │
│  更新学习率                                           │
│  optimizer.step()                                    │
│  zero_grad(set_to_none=True)                         │
└──────────────────────────────────────────────────────┘
```

## 13. 面试常见问题

### Q1: 为什么需要梯度累积?

**答**:
- 大 batch 提升训练稳定性，但显存装不下
- 梯度累积：多次小 batch forward/backward，累积梯度后一次更新
- 数学上等价于大 batch，但显存只需要小 batch 的量
- 注意 loss 要除以 grad_accum_steps

---

### Q2: 为什么用 BF16 而不是 FP16?

**答**:
- BF16 数值范围和 FP32 一样大 (8 位指数)
- 不容易溢出，不需要 GradScaler
- 代码更简单，训练更稳定
- 精度略低于 FP16 (7 位尾数 vs 10 位)，但实践中够用

---

### Q3: 梯度裁剪应该在什么时候做?

**答**:
- 在梯度累积**完成后**，optimizer.step() **之前**
- 顺序: backward × N → clip → step → zero_grad
- 返回的 grad_norm 是裁剪前的值，用于监控

---

### Q4: zero_grad(set_to_none=True) 和 zero_grad() 的区别?

**答**:
- `zero_grad()`: grad 设为全零 tensor (占显存)
- `set_to_none=True`: grad 设为 None (释放显存)
- 后者更快更省显存，但下次访问 `.grad` 会是 None

---

### Q5: 如何判断训练是否正常?

**答**:
1. **初始 loss**: 应该接近 ln(vocab_size)，否则初始化有问题
2. **loss 下降**: 前几百步应该快速下降
3. **grad_norm**: 在合理范围 (0.1-10)，偶尔 spike 正常
4. **MFU**: > 40% 说明 GPU 利用率正常
5. **val loss**: 应该和 train loss 趋势一致，差距不大

---

### Q6: torch.compile 的 dynamic=False 是什么意思?

**答**:
- `dynamic=False`: 告诉编译器输入形状固定不变
- 编译器可以做更激进的优化 (shape 特化的 kernel)
- 训练时 batch_size 和 seq_len 固定，可以用
- 推理/评估时输入形状可能变化，用原始未编译模型

---

### Q7: 检查点不保存优化器状态会怎样?

**答**:
- Adam 的 m (动量) 和 v (方差) 从 0 开始
- 恢复后相当于 "冷启动"，梯度估计不准
- 需要重新 warmup，否则可能发散
- 训练不连续，结果可能和不中断时不同

## 14. 总结速查表

| 主题 | 要点 |
|------|------|
| **损失函数** | CrossEntropy，初始 loss ≈ ln(vocab_size) |
| **梯度累积** | loss 除以 accum_steps，等效大 batch |
| **混合精度** | BF16 autocast，不需要 GradScaler |
| **梯度裁剪** | 累积完成后，step 之前，默认 max_norm=1.0 |
| **zero_grad** | set_to_none=True 更快更省显存 |
| **Prefetch** | backward 时 CPU 加载下一个 batch |
| **torch.compile** | dynamic=False，保留 orig_model 用于推理 |
| **检查点** | 必须保存 model + optimizer + dataloader + step |
| **MFU** | > 40% 正常，< 20% 有瓶颈 |

### 单步训练的完整顺序

```
1. 调整 lr (schedule)
2. for micro_step in accum_steps:
       autocast → forward → loss/accum → backward → prefetch
3. clip_grad_norm_
4. optimizer.step()
5. zero_grad(set_to_none=True)
```